# 03b Berlin Optimization

Hyperparameter tuning and final training of ML and NN champions on Berlin data.

**Inputs:**
- `algorithm_comparison.json` (from exp_10)
- Berlin train/val/test splits (from 03a)

**Outputs:**
- `hp_tuning_ml.json` - Optuna results for ML champion
- `hp_tuning_nn.json` - Optuna results for NN champion
- `berlin_ml_champion.pkl` - Trained ML model
- `berlin_nn_champion.pt` - Trained NN model
- `berlin_scaler.pkl` - Feature scaler
- `label_encoder.pkl` - Label encoder
- `berlin_evaluation.json` - Test metrics
- Error analysis figures (confusion matrix, feature importance, etc.)

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import pickle
import time
import datetime
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'models').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    evaluation_analysis,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

start_time = time.time()
config = load_experiment_config()
print(f"Config loaded. Random seed: {config['global']['random_seed']}")

In [ ]:
# Load algorithm comparison results
algo_path = OUTPUT_DIR / 'metadata/algorithm_comparison.json'
algorithm_comparison = validate_algorithm_comparison(algo_path)

ml_name = algorithm_comparison['ml_champion']['algorithm']
nn_name = algorithm_comparison['nn_champion']['algorithm']

print(f"ML Champion: {ml_name}")
print(f"NN Champion: {nn_name}")

# Load Berlin splits
train_df, val_df, test_df = data_loading.load_berlin_splits(DATA_DIR / 'phase_3_experiments')
feature_cols = data_loading.get_feature_columns(train_df)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_df)}")
print(f"  Val: {len(val_df)}")
print(f"  Test: {len(test_df)}")
print(f"  Features: {len(feature_cols)}")

# Preprocessing
x_train = train_df[feature_cols].to_numpy(dtype=float)
x_val = val_df[feature_cols].to_numpy(dtype=float)
x_test = test_df[feature_cols].to_numpy(dtype=float)

x_train_scaled, x_val_scaled, x_test_scaled, scaler = preprocessing.scale_features(
    train_df[feature_cols], x_val=val_df[feature_cols], x_test=test_df[feature_cols]
)

y_train, label_to_idx, idx_to_label = preprocessing.encode_genus_labels(train_df['genus_latin'])
y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()
y_test = test_df['genus_latin'].map(label_to_idx).to_numpy()

# Combine train+val for final training
x_combined_scaled = np.vstack([x_train_scaled, x_val_scaled])
y_combined = np.concatenate([y_train, y_val])

print(f"  Combined (train+val): {len(x_combined_scaled)}")

# Save label encoder separately (M4)
label_encoder_data = {'label_to_idx': label_to_idx, 'idx_to_label': idx_to_label}
with (OUTPUT_DIR / 'models/label_encoder.pkl').open('wb') as f:
    pickle.dump(label_encoder_data, f)
print(f"\nSaved label_encoder.pkl")

In [ ]:
# ML Champion: Optuna HP Tuning (C2)
print(f"\n{'='*60}")
print(f"ML CHAMPION: {ml_name}")
print(f"{'='*60}")

ml_search_space = config['hp_tuning']['search_spaces'][ml_name]

# Create Optuna study
ml_study = hp_tuning.create_study(
    direction='maximize',
    random_seed=config['global']['random_seed'],
)

# Build objective with spatial CV
cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])

ml_objective = hp_tuning.build_objective(
    model_name=ml_name,
    x=x_train_scaled,
    y=y_train,
    groups=train_df['block_id'].values,
    cv=cv,
    search_space=ml_search_space,
)

# Run optimization
print(f"\nRunning Optuna HP tuning ({config['hp_tuning']['optuna']['n_trials']} trials)...")
ml_hp_results = hp_tuning.run_optuna_search(
    ml_study,
    ml_objective,
    n_trials=config['hp_tuning']['optuna']['n_trials'],
    timeout_seconds=config['hp_tuning']['optuna']['timeout_seconds'],
)

# Save HP tuning results
ml_hp_path = OUTPUT_DIR / 'metadata/hp_tuning_ml.json'
ml_hp_path.write_text(json.dumps(ml_hp_results, indent=2))
validate_hp_tuning_result(ml_hp_path)

print(f"\nBest ML hyperparameters:")
for k, v in ml_hp_results['best_params'].items():
    print(f"  {k}: {v}")
print(f"  CV F1: {ml_hp_results['best_value']:.4f}")

In [ ]:
# Train final ML model with best params
ml_best_params = ml_hp_results['best_params']
ml_model = models.create_model(ml_name, model_params=ml_best_params)

print(f"\nTraining final ML model on train+val...")
ml_model = training.train_final_model(
    ml_model,
    x_combined_scaled,
    y_combined,
)

# Save ML model
ml_model_path = OUTPUT_DIR / 'models/berlin_ml_champion.pkl'
training.save_model(
    ml_model,
    ml_model_path,
    metadata={
        'model_name': ml_name,
        'feature_columns': feature_cols,
        'label_to_idx': label_to_idx,
        'idx_to_label': idx_to_label,
        'best_params': ml_best_params,
        'hp_tuning_cv_f1': ml_hp_results['best_value'],
    },
)

print(f"Saved {ml_model_path}")

In [ ]:
# NN Champion: Optuna HP Tuning (C3)
print(f"\n{'='*60}")
print(f"NN CHAMPION: {nn_name}")
print(f"{'='*60}")

nn_search_space = config['hp_tuning']['search_spaces'][nn_name]

# Add structural params for CNN1D
if nn_name == 'cnn_1d':
    # Infer structural params from feature columns
    n_temporal_bases = len([f for f in feature_cols if not f.startswith('CHM')])
    n_months = 12  # Assuming 12 months
    n_static = len([f for f in feature_cols if f.startswith('CHM')])
    
    print(f"\nCNN1D structure:")
    print(f"  n_temporal_bases: {n_temporal_bases}")
    print(f"  n_months: {n_months}")
    print(f"  n_static_features: {n_static}")
    print(f"  n_classes: {len(label_to_idx)}")

# Create NN study
nn_study = hp_tuning.create_study(
    direction='maximize',
    random_seed=config['global']['random_seed'],
)

nn_objective = hp_tuning.build_objective(
    model_name=nn_name,
    x=x_train_scaled,
    y=y_train,
    groups=train_df['block_id'].values,
    cv=cv,
    search_space=nn_search_space,
)

print(f"\nRunning Optuna HP tuning ({config['hp_tuning']['optuna']['n_trials']} trials)...")
nn_hp_results = hp_tuning.run_optuna_search(
    nn_study,
    nn_objective,
    n_trials=config['hp_tuning']['optuna']['n_trials'],
    timeout_seconds=config['hp_tuning']['optuna']['timeout_seconds'],
)

# Save NN HP results
nn_hp_path = OUTPUT_DIR / 'metadata/hp_tuning_nn.json'
nn_hp_path.write_text(json.dumps(nn_hp_results, indent=2))
validate_hp_tuning_result(nn_hp_path)

print(f"\nBest NN hyperparameters:")
for k, v in nn_hp_results['best_params'].items():
    print(f"  {k}: {v}")
print(f"  CV F1: {nn_hp_results['best_value']:.4f}")

In [ ]:
# Train final NN model with best params
nn_best_params = nn_hp_results['best_params'].copy()

# Add structural params for CNN1D
if nn_name == 'cnn_1d':
    nn_best_params.update({
        'n_temporal_bases': n_temporal_bases,
        'n_months': n_months,
        'n_static_features': n_static,
        'n_classes': len(label_to_idx),
    })

nn_model = models.create_model(nn_name, model_params=nn_best_params)

print(f"\nTraining final NN model on train+val...")
if nn_name == 'cnn_1d':
    # Extract fit params for CNN training
    fit_params = {
        'batch_size': nn_best_params.get('batch_size', 64),
        'epochs': nn_best_params.get('epochs', 50),
        'learning_rate': nn_best_params.get('learning_rate', 0.001),
        'early_stopping_patience': nn_best_params.get('early_stopping_patience', 10),
    }
    nn_model = training.train_final_model(nn_model, x_combined_scaled, y_combined, fit_params=fit_params)
else:
    # TabNet or other
    nn_model = training.train_final_model(nn_model, x_combined_scaled, y_combined)

# Save NN model
nn_model_path = OUTPUT_DIR / 'models/berlin_nn_champion.pt'
training.save_model(
    nn_model,
    nn_model_path,
    metadata={
        'model_name': nn_name,
        'feature_columns': feature_cols,
        'label_to_idx': label_to_idx,
        'idx_to_label': idx_to_label,
        'best_params': nn_best_params,
        'hp_tuning_cv_f1': nn_hp_results['best_value'],
    },
)

print(f"Saved {nn_model_path}")

In [ ]:
# Save scaler
scaler_path = OUTPUT_DIR / 'models/berlin_scaler.pkl'
with scaler_path.open('wb') as f:
    pickle.dump(scaler, f)
print(f"\nSaved {scaler_path}")

In [ ]:
# Evaluate ML model on test set
print(f"\n{'='*60}")
print(f"EVALUATION")
print(f"{'='*60}")

ml_preds = ml_model.predict(x_test_scaled)
ml_metrics = evaluation.compute_metrics(y_test, ml_preds)
ml_per_class = evaluation.compute_per_class_metrics(y_test, ml_preds, list(idx_to_label.values()))
ml_cm = evaluation.compute_confusion_matrix(y_test, ml_preds, labels=list(range(len(idx_to_label))))

print(f"\nML Champion ({ml_name}) Test Results:")
for metric, value in ml_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Save evaluation results
eval_data = {
    'ml_champion': ml_name,
    'nn_champion': nn_name,
    'metrics': ml_metrics,
    'per_class': ml_per_class.to_dict(orient='records'),
    'confusion_matrix': ml_cm.tolist(),
}

metrics_path = OUTPUT_DIR / 'metadata/berlin_evaluation.json'
metrics_path.write_text(json.dumps(eval_data, indent=2))
validate_evaluation_metrics(metrics_path)
print(f"\nSaved {metrics_path}")

In [ ]:
# Post-training error analysis (H4)
print(f"\n{'='*60}")
print(f"ERROR ANALYSIS")
print(f"{'='*60}")

fig_dir = OUTPUT_DIR / 'figures/berlin_optimization'
fig_dir.mkdir(parents=True, exist_ok=True)
class_labels = list(idx_to_label.values())

# 6a: Confusion matrix
print(f"\nGenerating confusion matrix...")
visualization.plot_confusion_matrix(
    ml_cm,
    class_labels,
    output_path=fig_dir / 'confusion_matrix.png',
)

# 6b: Per-genus F1 scores
print(f"Generating per-genus F1 plot...")
per_genus_f1 = {row['genus']: row['f1_score'] for row in ml_per_class.to_dict(orient='records')}
visualization.plot_per_genus_f1(
    per_genus_f1,
    output_path=fig_dir / 'per_genus_f1.png',
)

# 6c: Worst confused pairs
print(f"Analyzing worst confused pairs...")
confused_pairs = evaluation.analyze_worst_confused_pairs(
    y_test,
    ml_preds,
    class_labels,
    genus_german_map=None,
    top_n=10,
)

print(f"\nTop 5 confused pairs:")
for pair in confused_pairs[:5]:
    print(f"  {pair['true_genus']} → {pair['pred_genus']}: {pair['count']} errors")

# 6d: Conifer vs Deciduous
print(f"\nAnalyzing conifer vs deciduous...")
cd_metrics = evaluation.analyze_conifer_deciduous(
    y_test,
    ml_preds,
    class_labels,
    config['genus_groups'],
)

print(f"  Conifer F1: {cd_metrics['conifer_f1']:.4f} (n={cd_metrics['conifer_n']})")
print(f"  Deciduous F1: {cd_metrics['deciduous_f1']:.4f} (n={cd_metrics['deciduous_n']})")

# 6e: Metadata impact (tree_type) - H4 addition
print(f"\nAnalyzing tree type impact...")
if 'tree_type' in test_df.columns:
    tree_type_analysis = evaluation.analyze_by_metadata(
        y_test,
        ml_preds,
        class_labels,
        test_df['tree_type'],
        metadata_name='tree_type',
    )
    
    print(f"  Performance by tree type:")
    for tree_type, metrics in tree_type_analysis.items():
        print(f"    {tree_type}: F1={metrics['f1_score']:.4f}, n={metrics['n_samples']}")
else:
    print(f"  tree_type column not available in test data")

# 6f: Plant year impact (decade) - H4 addition
print(f"\nAnalyzing plant year impact...")
if 'plant_year' in test_df.columns:
    decade_bins = evaluation.bin_plant_years(test_df['plant_year'])
    decade_analysis = evaluation.analyze_by_metadata(
        y_test,
        ml_preds,
        class_labels,
        decade_bins,
        metadata_name='plant_decade',
    )
    
    print(f"  Performance by plant decade:")
    for decade, metrics in decade_analysis.items():
        print(f"    {decade}: F1={metrics['f1_score']:.4f}, n={metrics['n_samples']}")
else:
    print(f"  plant_year column not available in test data")

# 6g: Species breakdown for low F1 genera - H4 addition
print(f"\nAnalyzing species breakdown for low F1 genera...")
if 'species_latin' in test_df.columns:
    species_breakdown = evaluation.analyze_species_breakdown(
        y_test,
        ml_preds,
        class_labels,
        test_df['species_latin'],
        genus_german_map=None,
        f1_threshold=0.50,
    )
    
    if species_breakdown:
        print(f"  Found {len(species_breakdown)} genera with F1 < 0.50")
        for genus, species_df in list(species_breakdown.items())[:3]:
            print(f"    {genus}: {len(species_df)} species analyzed")
    else:
        print(f"  All genera have F1 >= 0.50")
else:
    print(f"  species_latin column not available in test data")

# 6h: Feature importance
print(f"\nComputing feature importance...")
if hasattr(ml_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': ml_model.feature_importances_,
    }).sort_values('importance', ascending=False)
    
    visualization.plot_feature_importance(
        importance_df.head(30),
        output_path=fig_dir / 'feature_importance.png',
    )
    
    print(f"  Top 5 features:")
    for _, row in importance_df.head(5).iterrows():
        print(f"    {row['feature']}: {row['importance']:.4f}")

print(f"\nError analysis complete. Figures saved to {fig_dir}")

In [ ]:
# Execution log (L2)
elapsed = time.time() - start_time

log = {
    'status': 'completed',
    'timestamp': datetime.datetime.now().isoformat(),
    'runtime_seconds': elapsed,
    'config_hash': hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest(),
    'ml_champion': ml_name,
    'nn_champion': nn_name,
    'ml_hp_tuning': {
        'n_trials': ml_hp_results['n_trials'],
        'best_cv_f1': ml_hp_results['best_value'],
    },
    'nn_hp_tuning': {
        'n_trials': nn_hp_results['n_trials'],
        'best_cv_f1': nn_hp_results['best_value'],
    },
    'test_metrics': ml_metrics,
}

log_path = OUTPUT_DIR / 'logs/03b_berlin_optimization.json'
log_path.write_text(json.dumps(log, indent=2))

print(f"\n{'='*60}")
print(f"COMPLETED")
print(f"{'='*60}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Log saved: {log_path}")
print(f"\nOutputs:")
print(f"  ML model: {ml_model_path}")
print(f"  NN model: {nn_model_path}")
print(f"  Scaler: {scaler_path}")
print(f"  Label encoder: models/label_encoder.pkl")
print(f"  Metrics: {metrics_path}")
print(f"  Figures: {fig_dir}")